# (IIP314W-2) [2026-T2] OPTIMIZACIÓN APLICADA A NEGOCIOS
## Ayudantía 7: Modelamiento con Big-M, análisis de sensibilidad y precios sombra

---

**Profesor:** Rodrigo Trigo Vilches

**Ayudante:** Vicente Ramírez

**Fecha:** 5 de agosto de 2026

**Universidad del Desarrollo**

---

## Objetivos

- Formular un modelo **entero-mixto (MIP)** compacto que combine cuatro estructuras clásicas: **proporciones** de mezcla, **balance de inventario**, una **restricción de activación Big-M** y capacidad de transporte en **masa y volumen**.
- Resolverlo en `gurobipy` e interpretar el plan en términos de negocio.
- Hacer **análisis de sensibilidad de los recursos**: precios sombra, holgura complementaria y **rango de validez**.
- Entender por qué un MIP **no tiene precios sombra** y cómo se obtienen **fijando la variable binaria en su valor óptimo** para transformarlo en un LP.
- Distinguir cuándo el precio sombra **responde** una pregunta de negocio y cuándo hay que **volver a resolver**.

> **Conexión con las clases.** En las Clases 22 y 27 el profesor construyó el dual, el teorema de holguras complementarias y el análisis de sensibilidad ($\partial z^*/\partial b_i$, rangos). En la Ayudantía 6 leímos los precios sombra **en el tableau óptimo**. Aquí hacemos lo mismo sobre un modelo con una **binaria**, donde el tableau ya no basta.

## Sección 0 — Repaso

### 0.1 Proporciones de mezcla

Para exigir que el origen $g$ no aporte más de una fracción $\bar\alpha_{gi}$ del blend $i$, la tentación es escribir

$$\frac{x_{gi}}{\sum_{g'} x_{g'i}}\le\bar\alpha_{gi},$$

que **no es lineal** (variable en el denominador) y además está indefinida si no se produce nada. Se **multiplica cruzado**:

$$\boxed{\;x_{gi}\;\le\;\bar\alpha_{gi}\sum_{g'} x_{g'i}\;}$$

Lineal, y si el blend no se produce fuerza correctamente $x_{gi}=0$.

### 0.2 Balance de inventario

$$\text{inventario al cierre de }t\;=\;\text{inventario al cierre de }t-1\;+\;\text{producción}_t\;-\;\text{despachos}_t$$

Es una **igualdad**, y el primer periodo usa el **dato** $r^0_i$ (la variable $r_{i,0}$ no existe).

### 0.3 Activación Big-M

Queremos: *"la tostadora rinde $K$ kg por semana; se puede sumar un turno extra de $K^{ex}$ kg, pero habilitarlo cuesta $F$"*. Con $y_t\in\{0,1\}$:

$$\boxed{\;\sum_{g}\sum_i x_{git}\;\le\;K+M\,y_t\;}\qquad\text{y en la objetivo }\;-F\,y_t$$

- $y_t=0$ ⟹ el tope es $K$: el turno está apagado.
- $y_t=1$ ⟹ el tope es $K+M$: hay que elegir $M$ **igual a la capacidad extra real**, $M=K^{ex}$.

La lógica *"si uso, pago"* la induce el óptimo: como $F>0$, el modelo nunca pone $y_t=1$ sin necesitarlo.

**¿Y si ponemos $M=10^9$?** La restricción sigue siendo *correcta* pero la **relajación lineal** se arruina: el LP puede tostar 3.000 kg con $y_t=3\cdot10^{-6}$ y pagar casi nada de costo fijo. La cota del *branch and bound* queda floja y el modelo se vuelve lento. **Regla: el $M$ más chico que siga siendo válido.**

### 0.4 Masa y volumen: dos restricciones, no una

Un camión tiene **dos** límites y basta que uno se agote: kilos ($W$) y metros cúbicos ($V^{cam}$):

$$\sum_i v_{it}\le W\,\qquad\text{(masa)}\qquad\qquad \sum_i \nu_i\,v_{it}\le V^{cam}\qquad\text{(volumen)}$$

con $\nu_i$ el **volumen específico** [m³/kg]. Escribir solo una es un error clásico. Cuál de las dos aprieta **no lo decidimos nosotros**: lo dice el precio sombra.

### 0.5 Precios sombra, rango de validez y el problema de los MIP

Para un LP, el precio sombra de la restricción $i$ es su variable dual: $\pi_i=\partial z^*/\partial b_i$, el valor marginal de una unidad extra del recurso. Tres propiedades:

| Propiedad | Enunciado | Lectura de negocio |
|:--|:--|:--|
| **Holgura complementaria** | $\pi_i\cdot\text{Slack}_i=0$ | si sobra recurso no pago por más; si pago por más, es porque está agotado |
| **Rango de validez** | $\pi_i$ es constante solo mientras $b_i\in[b_i^{\text{low}},b_i^{\text{up}}]$ | el precio vale para **cambios chicos** |
| **Signo** | en un máximo, $\pi_i\ge0$ para una restricción $\le$ | relajar un techo nunca empeora |

**Un MIP no tiene precios sombra.** La dualidad es teoría de programación **lineal**: vive en el tableau del Simplex. Mientras haya binarias libres, $z^*(b)$ no es cóncava ni lineal por tramos y las duales no están definidas. La técnica estándar es en dos pasos:

1. Resolver el **MIP** y quedarse con el óptimo $y^*$ de la binaria.
2. **Fijar $y^*$** y tratarla como **dato**. Queda un **LP puro** en las variables continuas, con el mismo valor óptimo, y ahí sí hay duales: `.Pi`, `.Slack`, `.RC`, `.SARHSLow`, `.SARHSUp`.

En `gurobipy` esto se obtiene con `m.fixed()`, o reconstruyendo el modelo con $y^*$ como número.

> **Y la advertencia clave:** esos duales valen **manteniendo fija la decisión discreta**. Sirven para *"¿cuánto vale un m³ más de bodega?"*; **no** sirven para *"¿me conviene arrendar otro camión?"* o *"¿conviene habilitar el turno extra?"*, porque eso cambia $y$ o cambia el problema. Ahí hay que **volver a resolver**.

---

# Ejercicio — Tostaduría "Kütral Café"

### Contexto

**Kütral Café** es una tostaduría de especialidad que vende café en grano a cafeterías. Hay que planificar las próximas **tres semanas**.

Se compra café verde de dos orígenes, **arábica de Colombia** (COL) y **robusta de Vietnam** (VIE), a precio $c_g$ por kilo. Con ellos se preparan dos blends: **Espresso Cumbre** (ESP) e **Intenso Patagón** (INT). Cada blend es una **receta**: la robusta —más barata— no puede superar una fracción $\bar\alpha_{gi}$ de la mezcla, o el perfil de taza se pierde.

La **tostadora** procesa hasta $K$ kg por semana. Si no alcanza, se puede **habilitar un turno extra de noche** que agrega $K^{ex}$ kg, pero cuesta un **fijo $F$** por semana (personal nocturno, supervisión). Se decide semana a semana.

Lo producido y no despachado queda en la **bodega**, limitada por **volumen** $V^{bod}$ (m³), no por peso: el espacio es lo escaso. Almacenar cuesta $h$ por kg y por semana.

Los despachos salen en camión: hasta $W$ kg **y** $V^{cam}$ m³ por camión, a un costo $ct$ por camión-semana. Aquí importa el **envase**: el Espresso Cumbre va en **bolsas de 250 g** para góndola y el Intenso Patagón en **sacos de 1 kg** para cafeterías, así que el ESP ocupa **casi el doble de volumen por kilo** que el INT.

Cada semana hay un **compromiso contractual** $\underline d_{it}$ que debe cumplirse y una **demanda máxima** $\bar d_{it}$ que el mercado absorbe. Se vende a precio $pr_i$.

**Objetivo:** maximizar el margen de las tres semanas (ingresos − café verde − costo fijo del turno extra − almacenamiento − transporte).

### Conjuntos y parámetros

| Símbolo | Descripción |
|:--|:--|
| $\mathcal{G}=\{\text{COL},\text{VIE}\}$ | orígenes de café verde (índice $g$) |
| $\mathcal{I}=\{\text{ESP},\text{INT}\}$ | blends (índice $i$) |
| $\mathcal{T}=\{1,2,3\}$ | semanas (índice $t$) |

| Símbolo | Significado | Unidad |
|:--|:--|:--|
| $c_g$ | costo del café verde del origen $g$ | \$/kg |
| $pr_i$ | precio de venta del blend $i$ | \$/kg |
| $\bar\alpha_{gi}$ | fracción máxima del origen $g$ en la receta del blend $i$ | — |
| $K$ | capacidad de tostado del turno normal | kg/semana |
| $K^{ex}$ | capacidad adicional del turno extra | kg/semana |
| $F$ | costo fijo de habilitar el turno extra | \$/semana |
| $h$ | costo de almacenamiento | \$/(kg·semana) |
| $V^{bod}$ | volumen de la bodega | m³ |
| $\nu_i$ | volumen específico del blend $i$ (según su envase) | m³/kg |
| $W,\ V^{cam}$ | capacidad de un camión en masa y en volumen | kg, m³ |
| $nc_t$ | camiones disponibles en la semana $t$ | camiones |
| $ct$ | costo por camión-semana | \$ |
| $\underline d_{it},\ \bar d_{it}$ | compromiso contractual y demanda máxima | kg |
| $r^0_i$ | inventario inicial | kg |

### Datos de la instancia

Costos y precios en **\$ mil**.

| | COL | VIE | | | ESP | INT |
|:--|--:|--:|:--|:--|--:|--:|
| $c_g$ (\$ mil/kg) | 6,0 | 3,5 | | $pr_i$ (\$ mil/kg) | 14,0 | 11,0 |
| $\bar\alpha_{g,\text{ESP}}$ | 1,00 | **0,25** | | $\nu_i$ (m³/kg) | **0,0045** | **0,0025** |
| $\bar\alpha_{g,\text{INT}}$ | 1,00 | **0,60** | | envase | bolsas 250 g | sacos 1 kg |

| Escalar | $K$ | $K^{ex}$ | $F$ | $h$ | $V^{bod}$ | $W$ | $V^{cam}$ | $ct$ | $nc_t$ |
|:--|--:|--:|--:|--:|--:|--:|--:|--:|--:|
| valor | 1.500 | 400 | 500 | 0,10 | 1,1 | 2.400 | 7,0 | 600 | 1 |

| Demanda (kg) | $t=1$ | $t=2$ | $t=3$ |
|:--|--:|--:|--:|
| $\bar d_{\text{ESP},t}$ (máxima) | 700 | 1.200 | 1.100 |
| $\bar d_{\text{INT},t}$ (máxima) | 300 | 600 | 1.200 |
| $\underline d_{\text{ESP},t}$ (compromiso) | 280 | 480 | 440 |
| $\underline d_{\text{INT},t}$ (compromiso) | 120 | 240 | 480 |

Inventario inicial $r^0_i=0$.

### Se pide

**Parte A — Formulación (a mano, sin números).**

1. Defina las **variables de decisión** con su dominio, la **función objetivo** y las **seis familias de restricciones** en notación de sumatorias con su $\forall$.
2. Justifique el valor de $M$ en la restricción de activación. ¿Qué pasaría con $M=10^9$?
3. ¿Por qué la restricción de proporciones **no** se escribe como un cociente? ¿Por qué la masa y el volumen del camión son **dos** restricciones y no una?

**Parte B — Resolución en Gurobi.**

4. Implemente el modelo y resuélvalo. Reporte el margen óptimo, en qué semanas se habilita el turno extra, el plan de producción y despacho, y la trayectoria del inventario.
5. **Interprete el plan**: ¿por qué se produce por adelantado y se guarda inventario? ¿por qué el turno extra se habilita solo en una semana? ¿queda demanda sin atender, y quién la está bloqueando?

**Parte C — Análisis de sensibilidad.**

6. El modelo es un MIP. **Fije la binaria en su óptimo** para obtener el LP equivalente; verifique que el valor óptimo coincide y que ya no es un MIP.
7. Construya la tabla de **precios sombra** de los cuatro recursos (tostado, bodega, masa del camión, volumen del camión) con holgura, lado derecho y **rango de validez**. Verifique la **holgura complementaria**.
8. Los dos recursos activos de la semana 3 se pueden calcular **a mano**: escriba las dos restricciones duales activas (una por blend) y resuelva el sistema $2\times2$. Compare con lo que reporta Gurobi.
9. Interprete el precio sombra de la **bodega**: descompóngalo en los parámetros del problema (debe cuadrar exacto).
10. ¿Cuánto vale ampliar la bodega en $0{,}05$ m³? ¿Y en $0{,}4$ m³? ¿Y agrandar la caja del camión en $0{,}1$ m³? ¿Y en $1{,}0$ m³? **Compare la predicción del precio sombra con el resultado de volver a resolver** y explique con el rango de validez. Grafique $z^*(b)$ para ambos recursos.
11. ¿Por qué el precio sombra de la **masa** del camión es cero? ¿Qué le recomendaría a la empresa si le ofrecen un camión con 600 kg más de carga útil?
12. Arrendar un **segundo camión** en la semana 3 cuesta $ct=600$. ¿Conviene? ¿Puede responderlo con el precio sombra? Resuelva de nuevo y compare.
13. ¿Conviene habilitar el turno extra en las semanas 1 o 2? Responda **forzando la binaria** y comparando, y explique por qué esta pregunta no se puede responder con los duales.

---

## Parte A — Formulación

_Plantee el modelo de forma **algebraica** (sumatorias, parámetros simbólicos, sin números): variables con su dominio, función objetivo y las seis familias de restricciones con su $\forall$. Responda también las preguntas 2 y 3._

---

## Parte B — Resolución en Gurobi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB, quicksum

### B.0 — Datos

_Se entregan. El modelo debe leerlos del diccionario: **ningún número escrito dentro del modelo**._

In [ ]:
def datos():
    """Instancia: 2 origenes x 2 blends x 3 semanas. Costos y precios en $ mil."""
    d = {}
    d["G"] = ["COL", "VIE"]                       # origenes de cafe verde
    d["I"] = ["ESP", "INT"]                       # blends
    d["T"] = [1, 2, 3]                            # semanas

    d["c"]  = {"COL": 6.0, "VIE": 3.5}            # costo del cafe verde ($ mil/kg)
    d["pr"] = {"ESP": 14.0, "INT": 11.0}          # precio de venta ($ mil/kg)
    d["amax"] = {("COL","ESP"): 1.00, ("VIE","ESP"): 0.25,   # fraccion maxima en la receta
                 ("COL","INT"): 1.00, ("VIE","INT"): 0.60}

    d["K"], d["Kex"], d["F"] = 1500, 400, 500     # tostado: normal, turno extra, costo fijo
    d["h"] = 0.10                                 # almacenamiento ($ mil/kg-semana)
    d["Vbod"] = 1.1                               # volumen de bodega (m3)
    d["vol"] = {"ESP": 0.0045, "INT": 0.0025}     # volumen especifico (m3/kg) segun envase

    d["W"], d["Vcam"], d["ct"] = 2400, 7.0, 600   # camion: kg, m3, $ mil por camion-semana
    d["nc"] = {1: 1, 2: 1, 3: 1}                  # camiones disponibles por semana (DATO)

    d["dmax"] = {("ESP",1):700, ("ESP",2):1200, ("ESP",3):1100,
                 ("INT",1):300, ("INT",2):600,  ("INT",3):1200}
    d["dmin"] = {k: round(0.4*val) for k, val in d["dmax"].items()}
    d["r0"] = {"ESP": 0.0, "INT": 0.0}
    return d

D = datos()
print("M = Kex =", D["Kex"], "kg   (la cota valida mas chica)")
print("Un camion se llena con", round(D["Vcam"]/D["vol"]["ESP"]), "kg de ESP  o",
      round(D["Vcam"]/D["vol"]["INT"]), "kg de INT  (carga util:", D["W"], "kg)")

### B.1 — El modelo

_Escriba **una sola** función que construya el **MIP** o el **LP con la binaria fija**, según el argumento `ybar`._

In [ ]:
# Implemente la funcion que construye el modelo.
#
#   def construir_modelo(d, ybar=None, **cambios):  ...  return m
#
# Pistas:
#   * ybar=None -> y_t binaria (vtype=GRB.BINARY).  ybar={t: 0/1} -> y_t es un NUMERO:
#     el costo fijo pasa a ser constante y el tope del tostado queda como lado derecho.
#   * los compromisos y la demanda maxima van como cotas:
#       v = m.addVars(I, T, lb=d["dmin"], ub=d["dmax"], name="v")
#   * ponga NOMBRE a cada familia (segundo argumento de addConstrs): receta, inventario,
#     tostado, bodega, peso, volumen. En la Parte C pedimos los duales por nombre.
#   * el primer periodo usa el dato d["r0"][i], no la variable del periodo 0.
#   * use m.Params.MIPGap = 0.0 : en la Parte C vamos a comparar optimos entre escenarios.
#### CÓDIGO AQUÍ ####

### B.2 — Resolución e interpretación (preguntas 4 y 5)

In [ ]:
# Resuelva el MIP e informe margen optimo, y*, plan de tostado, despachos
# (kg y m3 contra sus dos topes), inventario y receta efectiva de cada blend.
# Para cada semana compare  sum_i v[i,t]  con  W*nc_t   y   sum_i vol_i*v[i,t]  con  Vcam*nc_t:
#   ¿cual de los dos limites del camion es el que aprieta?
#### CÓDIGO AQUÍ ####

_Escriba aquí su interpretación: ¿por qué se produce por adelantado y qué blend se guarda (y por qué ése)? ¿por qué el turno extra se habilita solo en una semana? ¿queda demanda sin atender, y cuál de los dos límites del camión la está bloqueando?_

### B.3 — Verificación de factibilidad

In [ ]:
# Verifique numericamente las seis familias de restricciones sobre la solucion.
#### CÓDIGO AQUÍ ####

---

## Parte C — Análisis de sensibilidad

El modelo tiene **tres variables binarias** ($y_1,y_2,y_3$), así que es un MIP y **no tiene precios sombra**. Para hacer el análisis de sensibilidad lo convertimos en un **problema lineal fijando la binaria en su valor óptimo** $y^*=(0,0,1)$ — la misma técnica de la Tarea 4.

Lo hacemos por los dos caminos:

- **`m.fixed()`**: Gurobi devuelve el modelo con las enteras convertidas en continuas y fijadas ($lb=ub=y^*$).
- **Reconstruir el modelo** pasando $y^*$ como **número**. El óptimo es idéntico, pero ahora el lado derecho del tostado queda **numérico** ($K+K^{ex}y^*_t$), de modo que los rangos `SARHSLow`/`SARHSUp` se leen directamente en kilos.

Los tres modelos deben dar el **mismo valor objetivo**; la diferencia es que los dos últimos ya **no son MIP** (`IsMIP = 0`) y por lo tanto tienen duales.

In [ ]:
# (1) Guarde y* del MIP.
# (2) Obtenga el LP por los dos caminos y compare ObjVal e IsMIP:
#        fx = m.fixed();  fx.Params.OutputFlag = 0;  fx.optimize()
#        lp = construir_modelo(D, ybar=ystar);  lp.optimize()
# (3) Imprima el RHS de tostado[t] en el LP reconstruido: debe ser K + Kex*y*_t.
#### CÓDIGO AQUÍ ####

### Precios sombra de los recursos (pregunta 7)

In [ ]:
# Tabla de precios sombra de los recursos. Para cada restriccion:
#     c = lp.getConstrByName(nombre)  ->  c.Pi, c.Slack, c.RHS, c.SARHSLow, c.SARHSUp
# Familias a mirar: tostado[t], bodega[t], peso[t], volumen[t].
# Verifique la holgura complementaria: max |Pi*Slack| sobre las restricciones y
# max |RC*valor| sobre las variables con cota inferior 0 (ojo: v tiene lb > 0).
# Mire tambien el dual de receta[VIE,i,t]: ¿lo puede explicar con c_COL y c_VIE?
#### CÓDIGO AQUÍ ####

_Interprete cada precio sombra en términos de negocio. Preste atención a `tostado[2]` frente a `tostado[3]`: ¿por qué difieren exactamente en $h$?_

### C.8 — Los precios sombra de la semana 3, a mano (pregunta 8)

_En el óptimo se venden **los dos** blends en $t_3$, así que por holgura complementaria sus **dos restricciones duales están activas**. Cada kilo de blend $i$ consume $1$ kg de tostado y $\nu_i$ m³ de camión:_

$$\text{margen}_i=\pi_{\text{tostado}}+\nu_i\,\pi_{\text{volumen}}$$

_Escriba las dos ecuaciones, resuelva el sistema $2\times2$ y compare con Gurobi. Luego descomponga el precio sombra de la bodega (pregunta 9)._

In [ ]:
# Calcule a mano los precios sombra de la semana 3:
#   - margen unitario de cada blend con su receta optima (la robusta en su tope amax)
#   - las dos restricciones duales activas:  margen_i = pi_tostado + vol_i * pi_volumen
#   - resuelva el sistema 2x2 y compare con lo que reporta Gurobi.
# Luego descomponga el precio sombra de bodega[1]: ¿cuantos kg de que blend caben en 1 m3?
# ¿que capacidad libera cada kg adelantado, y cuanto cuesta guardarlo una semana?
#### CÓDIGO AQUÍ ####

### El rango de validez: hasta dónde se puede confiar (pregunta 10)

Aquí está el error más caro de la práctica: tomar el precio sombra y multiplicarlo por una cantidad grande.

In [ ]:
# Compare la prediccion del precio sombra con el resultado de RE-RESOLVER el MIP:
#   Vbod +0.05 | Vbod en el borde de su rango | Vbod +0.4 | Vbod +2.0
#   Vcam +0.1  | Vcam en el borde de su rango | Vcam +1.0 | W +600
# Pista: resolver(D, Vbod=1.15) usa el argumento **cambios de construir_modelo.
#### CÓDIGO AQUÍ ####

In [ ]:
# Grafique z*(b) para Vbod y para Vcam, marcando el punto base, la recta de pendiente Pi
# dentro de su rango de validez, y la extrapolacion invalida. Identifique los quiebres.
#### CÓDIGO AQUÍ ####

_¿Dentro del rango la predicción es exacta? ¿Y fuera? En el caso de la bodega la predicción sobreestima; en el del camión se queda corta. ¿Por qué un MIP puede fallar para los dos lados, si en un LP puro el precio sombra **siempre** sobreestima?_

### C.11 — ¿Y si nos ofrecen 600 kg más de carga útil? (pregunta 11)

_Responda con el precio sombra de `peso` y con el experimento $W=3.000$. Explique el resultado a partir del **envase** de cada blend. ¿Qué le pediría entonces al proveedor del camión? ¿Y por qué, a pesar de todo, valía la pena escribir la restricción de masa?_

### ¿Conviene arrendar un segundo camión en la semana 3? (pregunta 12)

Esta es la pregunta donde el precio sombra **no sirve**: un camión más cambia $nc_3$ de $1$ a $2$, o sea suma $7$ m³ de golpe — cuarenta veces el ancho del rango de validez.

In [ ]:
# Estime la ganancia del segundo camion con el precio sombra (Pi * Vcam - ct),
# luego resuelva de nuevo con nc={1:1, 2:1, 3:2} y compare. ¿Por que difieren tanto?
# ¿Cambio y*?
#### CÓDIGO AQUÍ ####

### ¿Conviene habilitar el turno extra en $t_1$ o $t_2$? (pregunta 13)

Tampoco se puede responder con los duales, y por una razón de fondo: **la binaria está fija en el LP**. Preguntar por $y_t$ es preguntar por otro problema entero. La respuesta se obtiene forzando la binaria y resolviendo.

In [ ]:
# Para cada semana, fuerce la binaria al valor contrario de y*_t y resuelva de nuevo:
#     mf = construir_modelo(D);  mf._var["y"][t].LB = 1   (o .UB = 0)
# Compare el margen contra el caso base. ¿Cuanto pierde/gana en cada caso?
# Compare tambien con lo que habria predicho el precio sombra de tostado[2] x 400 kg.
#### CÓDIGO AQUÍ ####

_¿Cuánto pierde o gana en cada caso? ¿Por qué el turno de $t_1$ se abre y **no se usa**? Compare la respuesta real con lo que habría predicho $\pi_{\text{tostado}[2]}\times400$ y explique la diferencia con el rango de validez._

---

## Para llevarse

| Estructura | Forma correcta |
|:--|:--|
| **Proporciones** | $x_{git}\le\bar\alpha_{gi}\sum_{g'}x_{g'it}$ — nunca como cociente |
| **Balance de inventario** | igualdad, con el dato $r^0_i$ en el primer periodo |
| **Activación Big-M** | $\sum\sum x_{git}\le K+M y_t$ con $M$ la cota válida **más chica** |
| **Masa y volumen** | **dos** restricciones distintas; el modelo decide cuál aprieta |
| **Sensibilidad en un MIP** | resolver → **fijar la binaria** → LP → `.Pi`, `.Slack`, `.SARHSLow/Up` |
| **Regla de oro** | un precio sombra sin su **rango de validez** no significa nada; y nunca se multiplica por un salto discreto |

---
*Ayudantía 7 — IIP314W Optimización Aplicada a Negocios — Universidad del Desarrollo — 2026-T2*